#### Exploratory and Data Analysis

##### Data Description

In [ ]:
#Split X_train, y_train
X_train = train_df.drop(['cat_id'], axis=1)
y_train = train_df['cat_id']
X_valid = valid_df.drop(['cat_id'], axis=1)
y_valid = valid_df['cat_id']
X_test = test_df.drop(['cat_id'], axis=1)
y_test = test_df['cat_id']

In [ ]:
# Training data information
print("=== TRAIN DATA INFO ===")
print(f"Train data shape: {train_df.shape}")
print(f"Columns: {train_df.columns}")
print(f"Data types: \n{train_df.dtypes}")
print(f"-1 values: \n{(train_df == -1).sum()}")

# Valid data information
print("\n=== TEST DATA INFO ===")
print(f"Train data shape: {valid_df.shape}")
print(f"Columns: {valid_df.columns}")
print(f"-1 values: \n{(valid_df == -1).sum()}")

# Testing data information
print("=== TEST DATA INFO ===")
print(f"Train data shape: {test_df.shape}")
print(f"Columns: {test_df.columns}")
print(f"-1 values: \n{(test_df == -1).sum()}")

In [ ]:
y_train.value_counts()

##### Data Visualization (For valid keypoint)

In [ ]:
missing_df = (X_train == -1)

In [ ]:
plt.figure(figsize=(12, 6))
sns.heatmap(missing_df, cbar=False, cmap='viridis')
plt.title("Missing Keypoints Pattern (Yellow = Missing(-1))")
plt.show()

df = train_df.copy()

df['missing_count'] = (df == -1).sum(axis=1)
sns.boxplot(data=df, x='cat_id', y='missing_count')
plt.title("Missing Points Count: Good (2) vs Bad (1)")
plt.show()

1) Since the missing value already replace with (-1,-1), check if any data has too much missing value

In [ ]:
value_count_df = train_df.copy()
X_columns = [col for col in value_count_df.columns if '_x' in col]
value_count_df['valid_points_count'] = (value_count_df[X_columns] != -1).sum(axis=1)

print(value_count_df['valid_points_count'].value_counts())

From above result, can see than there are some images has less than 10 keypoints.

In [ ]:
filtered_df = value_count_df[value_count_df['valid_points_count'] <= 11]

print("Images with 'valid_points_count' <= 10 and their COCO data:\n")

for index, row in filtered_df.iterrows():
    image_id = int(row['image_id'])
    valid_points_count = row['valid_points_count']

    # Find the corresponding image info in coco_data
    image_info = next((item for item in coco_data['images'] if item['id'] == image_id), None)

    if image_info:
        file_name = image_info['file_name']
        # Correct the image_path to include 'keyjoints_' as part of the filename
        image_path = os.path.join('/content/sitting-posture-keyjoint-dataset/keyjoint_train_dataset', 'keyjoints_' + file_name)
        print(f"Image ID: {image_id}")
        print(f"  File Name: {image_path}")
        print(f"  Valid Points Count: {valid_points_count:.2f}")
        try:
            img = Image.open(image_path)
            plt.figure(figsize=(8, 8)) # Create a new figure for each image
            plt.imshow(img)
            plt.title(f"Image ID: {image_id} - Valid Points Count: {valid_points_count:.2f}")
            plt.axis('off')
            plt.show()
        except FileNotFoundError:
            print(f"  Error: Annotated image not found at {image_path}")
        # You can add more details from row or image_info if needed
        print("------------------------------------------")
    else:
        print(f"Image ID: {image_id} - No matching image info found in coco_data.")
        print(f"  Valid Points Count: {valid_points_count:.2f}")
        print("------------------------------------------")

After checking, consider dropping images with less than 12 valid keypoint

##### Feature Distribution - Nose and shoulder distance

In [ ]:
df_valid = train_df[(train_df['nose_x'] != -1) & (train_df['left_shoulder_x'] != -1) & (train_df['right_shoulder_x'] != -1)].copy()

# Calculate shoulder center point
df_valid['sh_center_x'] = (df_valid['left_shoulder_x'] + df_valid['right_shoulder_x']) / 2

df_valid['head_forward'] = abs(df_valid['nose_x'] - df_valid['sh_center_x'])

plt.figure(figsize=(10, 5))
sns.kdeplot(data=df_valid, x='head_forward', hue='cat_id', fill=True)
plt.title("Head Forward Displacement: Good vs Bad")
plt.xlabel("Distance between Nose and Shoulder Center")
plt.show()

Display the "weird" image

In [ ]:
filtered_df = df_valid[df_valid['head_forward'] > 150]

print("Images with 'head_forward' > 150 and their COCO data:\n")

for index, row in filtered_df.iterrows():
    image_id = int(row['image_id'])
    head_forward_val = row['head_forward']

    # Find the corresponding image info in coco_data
    image_info = next((item for item in coco_data['images'] if item['id'] == image_id), None)

    if image_info:
        file_name = image_info['file_name']
        # Correct the image_path to include 'keyjoints_' as part of the filename
        image_path = os.path.join('/content/sitting-posture-keyjoint-dataset/keyjoint_train_dataset', 'keyjoints_' + file_name)
        print(f"Image ID: {image_id}")
        print(f"  File Name: {image_path}")
        print(f"  Head Forward: {head_forward_val:.2f}")
        try:
            img = Image.open(image_path)
            plt.figure(figsize=(8, 8)) # Create a new figure for each image
            plt.imshow(img)
            plt.title(f"Image ID: {image_id} - Head Forward: {head_forward_val:.2f}")
            plt.axis('off')
            plt.show()
        except FileNotFoundError:
            print(f"  Error: Annotated image not found at {image_path}")
        # You can add more details from row or image_info if needed
        print("------------------------------------------")
    else:
        print(f"Image ID: {image_id} - No matching image info found in coco_data.")
        print(f"  Head Forward: {head_forward_val:.2f}")
        print("------------------------------------------")

So: Consider dropping images where head_forward > 200

##### Feature Distribution - Left and Right Shoulder Distance

In [ ]:
df_shoulder_dist = train_df.copy()

# Calculate shoulder distance, handling cases where shoulder keypoints might be missing (-1)
# Replace -1 with NaN to exclude from calculation and ensure finite values
df_shoulder_dist['left_shoulder_x'] = df_shoulder_dist['left_shoulder_x'].replace(-1, np.nan)
df_shoulder_dist['left_shoulder_y'] = df_shoulder_dist['left_shoulder_y'].replace(-1, np.nan)
df_shoulder_dist['right_shoulder_x'] = df_shoulder_dist['right_shoulder_x'].replace(-1, np.nan)
df_shoulder_dist['right_shoulder_y'] = df_shoulder_dist['right_shoulder_y'].replace(-1, np.nan)

# Only calculate if both left and right shoulder points are present
df_shoulder_dist = df_shoulder_dist.dropna(subset=['left_shoulder_x', 'left_shoulder_y', 'right_shoulder_x', 'right_shoulder_y'])

df_shoulder_dist['sh_dist'] = ((df_shoulder_dist['left_shoulder_x'] - df_shoulder_dist['right_shoulder_x'])**2 +
                             (df_shoulder_dist['left_shoulder_y'] - df_shoulder_dist['right_shoulder_y'])**2)**0.5


plt.figure(figsize=(10, 6))
sns.kdeplot(data=df_shoulder_dist, x='sh_dist', hue='cat_id', fill=True, common_norm=False)
plt.title('Distribution of Shoulder Distance by Posture Category (Good vs. Bad)')
plt.xlabel('Shoulder Distance (pixels)')
plt.ylabel('Density')
plt.legend(title='Category', labels=['Bad Posture (1)', 'Good Posture (2)'])
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

In [ ]:
def is_unusual(row):
  image_id = int(row.image_id)
  sh_dist = ((row.left_shoulder_x - row.right_shoulder_x)**2 +
              (row.left_shoulder_y - row.right_shoulder_y)**2)**0.5

  if sh_dist > 300:
  #if sh_dist > 200 and sh_dist < 300:
    image_info = next((item for item in coco_data['images'] if item['id'] == image_id), None)
    file_name = image_info['file_name']
    image_path = os.path.join('/content/sitting-posture-keyjoint-dataset/keyjoint_train_dataset', 'keyjoints_' + file_name)
    img = Image.open(image_path)
    plt.figure(figsize=(8, 8)) # Create a new figure for each image
    plt.imshow(img)
    plt.title(f"Image ID: {image_id} - Shoulder distance: {sh_dist:.2f}")
    plt.axis('off')
    plt.show()
    return True

  return False

In [ ]:
for row in train_df.itertuples():
  if is_unusual(row):
    print(row)

All images that shoulder distance > 300 are unusual, consider drop it

In [ ]:
def is_unusual2(row):
  image_id = int(row["image_id"])
  sh_dist = ((row["left_shoulder_x"] - row["right_shoulder_x"])**2 +
              (row['left_shoulder_y'] - row['right_shoulder_y'])**2)**0.5

  if sh_dist < 1:
    image_info = next((item for item in coco_data['images'] if item['id'] == image_id), None)
    file_name = image_info['file_name']
    image_path = os.path.join('/content/sitting-posture-keyjoint-dataset/keyjoint_train_dataset', 'keyjoints_' + file_name)
    img = Image.open(image_path)
    plt.figure(figsize=(8, 8)) # Create a new figure for each image
    plt.imshow(img)
    plt.title(f"Image ID: {image_id} - Shoulder distance: {sh_dist:.2f}")
    plt.axis('off')
    plt.show()
    return True

  return False

In [ ]:
for index, row_data in train_df.iterrows():
  if is_unusual2(row_data):
    print(row_data)

All images that shoulder distance < 1 are unusual, consider drop it

##### Average Pose Visualization

In [ ]:
def plot_average_pose(df, title):
  """
  """
  temp = df.replace(-1, pd.NA).dropna(axis=0, how='any')
  mean_coord = temp.mean()

  connections = [
      ('nose_x', 'nose_y', 'left_eye_x', 'left_eye_y'),
      ('nose_x', 'nose_y', 'right_eye_x', 'right_eye_y'),
      ('left_ear_x', 'left_ear_y', 'left_eye_x', 'left_eye_y'),
      ('right_ear_x', 'right_ear_y', 'right_eye_x', 'right_eye_y'),
      ('left_shoulder_x', 'left_shoulder_y', 'right_shoulder_x', 'right_shoulder_y'),
      ('left_shoulder_x', 'left_shoulder_y', 'left_elbow_x', 'left_elbow_y'),
      ('right_shoulder_x', 'right_shoulder_y', 'right_elbow_x', 'right_elbow_y'),
      ('left_elbow_x', 'left_elbow_y', 'left_wrist_x', 'left_wrist_y'),
      ('right_elbow_x', 'right_elbow_y', 'right_wrist_x', 'right_wrist_y'),
      ('left_shoulder_x', 'left_shoulder_y', 'left_hip_x', 'left_hip_y'),
      ('right_shoulder_x', 'right_shoulder_y', 'right_hip_x', 'right_hip_y'),
      ('left_hip_x', 'left_hip_y', 'left_knee_x', 'left_knee_y'),
      ('right_hip_x', 'right_hip_y', 'right_knee_x', 'right_knee_y'),
      ('left_knee_x', 'left_knee_y', 'left_ankle_x', 'left_ankle_y'),
      ('right_knee_x', 'right_knee_y', 'right_ankle_x', 'right_ankle_y')]
  for x1, y1, x2, y2 in connections:
    plt.plot([mean_coord[x1], mean_coord[x2]], [mean_coord[y1], mean_coord[y2]], marker='o')
    plt.gca().invert_yaxis()
    plt.title(title)

plt.figure(figsize=(10, 10))
plt.subplot(1, 2, 1)
plot_average_pose(train_df[train_df['cat_id'] == 1], "Average Bad(1) Pose Visualization")
plt.subplot(1, 2, 2)
plot_average_pose(train_df[train_df['cat_id'] == 2], "Average Good(2) Pose Visualization")
plt.show()

From the mean image, can see that there are significant different posture between good and bad.<br>
To extract angles, consider:
1) Hips, Knees, Ankles
2) Hand, Elbow, Shoulder
3) Ear, Shoulder

##### Joint Scatter Plot - Using nose position

In [ ]:
df_nose = train_df[train_df['nose_x'] != -1]

plt.figure(figsize=(8, 8))
sns.scatterplot(data=df_nose, x='nose_x', y='nose_y', hue='cat_id', alpha=0.5, palette=['green', 'red'])
plt.gca().invert_yaxis()
plt.title("Scatter Cloud of Nose Positions")
plt.show()